# First TFLite Micro Inference - Model Training\n\n**Author:** suriyakumar P\n\nTrain a small light-level classifier and convert it to TensorFlow Lite for ESP32 TensorFlow Lite Micro inference.

In [ ]:
import numpy as np\nimport tensorflow as tf\nprint('TensorFlow:', tf.__version__)

In [ ]:
light = np.array([0.05, 0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.65, 0.69, 0.70, 0.80, 0.90, 1.00], dtype=np.float32)\nlabels = np.array([0,0,0,0,0,1,1,1,1,1,2,2,2,2], dtype=np.int32)\nprint('0=Dark, 1=Normal, 2=Bright')

In [ ]:
model = tf.keras.Sequential([\n    tf.keras.layers.Input(shape=(1,)),\n    tf.keras.layers.Dense(16, activation='relu'),\n    tf.keras.layers.Dense(3, activation='softmax')\n])\nmodel.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])\nmodel.fit(light, labels, epochs=300, verbose=0)\nprint('Training complete')

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)\ntflite_model = converter.convert()\nwith open('light_level_classifier.tflite', 'wb') as f:\n    f.write(tflite_model)\nprint('Created light_level_classifier.tflite')

In [ ]:
interpreter = tf.lite.Interpreter(model_path='light_level_classifier.tflite')\ninterpreter.allocate_tensors()\ninput_details = interpreter.get_input_details()\noutput_details = interpreter.get_output_details()\ninterpreter.set_tensor(input_details[0]['index'], np.array([[0.90]], dtype=np.float32))\ninterpreter.invoke()\nprediction = np.argmax(interpreter.get_tensor(output_details[0]['index'])[0])\nprint('0=Dark, 1=Normal, 2=Bright')\nprint('Predicted class:', prediction)

## ESP32 step\n\nAfter creating `light_level_classifier.tflite`, run `convert_model.py` locally to generate `model.h`, then place `model.h` beside `sketch.ino` in the Arduino ESP32 project.